# LeetCode 1535 — Find the Winner of an Array Game bằng Python

Link đề: https://leetcode.com/problems/find-the-winner-of-an-array-game/ · Độ khó: **Medium** · Chủ đề: Array, Simulation

Skeleton LeetCode:

```python
class Solution:
    def getWinner(self, arr: List[int], k: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô phỏng chính xác luật chơi bằng `list` và bằng `collections.deque`, hiểu vì sao `deque` nhanh hơn khi thao tác ở hai đầu.
2. Nhận ra điểm then chốt: sau tối đa `n − 1` vòng, **phần tử lớn nhất** đứng đầu và thắng mãi ⇒ `k` cực lớn (tới `10⁹`) không cần mô phỏng hết.
3. Rút gọn mô phỏng thành **một lượt duyệt mảng** `O(n)`, `O(1)` bộ nhớ: theo dõi "nhà vô địch hiện tại" và "chuỗi thắng".
4. Xử lý đúng chi tiết đếm chuỗi thắng (người thách đấu thắng thì chuỗi bắt đầu từ **1**, không phải 0).
5. Biết chứng minh tính đúng bằng invariant "mảng sau vòng `i` có dạng `[max(arr[0..i]), arr[i+1], …, arr[n−1], các phần tử thua…]`".

## 2. Đề bài & ví dụ

Cho mảng `arr` gồm các số nguyên **phân biệt** và số nguyên `k`. Trò chơi diễn ra giữa hai phần tử đầu `arr[0]` và `arr[1]`. Mỗi vòng, so sánh `arr[0]` với `arr[1]`: số **lớn hơn thắng và đứng ở vị trí 0**, số nhỏ hơn bị **chuyển xuống cuối mảng**. Trò chơi kết thúc khi một số thắng **`k` vòng liên tiếp**. Trả về số thắng cuộc (đề bảo đảm luôn có người thắng).

**Ví dụ 1:** `arr = [2,1,3,5,4,6,7]`, `k = 2` → `5`.

| Vòng | arr | Thắng | Chuỗi thắng |
|------|-----|-------|-------------|
| 1 | [2,1,3,5,4,6,7] | 2 | 1 |
| 2 | [2,3,5,4,6,7,1] | 3 | 1 |
| 3 | [3,5,4,6,7,1,2] | 5 | 1 |
| 4 | [5,4,6,7,1,2,3] | 5 | 2 |

**Ví dụ 2:** `arr = [3,2,1]`, `k = 10` → `3` (3 thắng 10 vòng đầu liên tiếp).

**Ràng buộc:** `2 ≤ arr.length ≤ 10⁵`, `1 ≤ arr[i] ≤ 10⁶`, các phần tử phân biệt, `1 ≤ k ≤ 10⁹`.

In [1]:
import bisect
import random
import time
from collections import deque
from typing import List

examples = [([2, 1, 3, 5, 4, 6, 7], 2, 5), ([3, 2, 1], 10, 3)]
for arr, k, want in examples:
    print(f"arr = {arr}, k = {k} -> mong đợi {want}")

arr = [2, 1, 3, 5, 4, 6, 7], k = 2 -> mong đợi 5
arr = [3, 2, 1], k = 10 -> mong đợi 3


Hai ví dụ thường gặp khác của bài (kiểm chứng bằng mô phỏng ở mục 5; nên đối chiếu lại đề gốc): `[1,9,8,2,3,7,6,4,5]`, `k = 7` → `9` và `[1,11,22,33,44,55,66,77,88,99]`, `k = 1000000000` → `99`.

In [2]:
more = [([1, 9, 8, 2, 3, 7, 6, 4, 5], 7, 9), ([1, 11, 22, 33, 44, 55, 66, 77, 88, 99], 10**9, 99)]
for arr, k, want in more:
    print(f"arr = {arr}, k = {k} -> mong đợi {want}")

arr = [1, 9, 8, 2, 3, 7, 6, 4, 5], k = 7 -> mong đợi 9
arr = [1, 11, 22, 33, 44, 55, 66, 77, 88, 99], k = 1000000000 -> mong đợi 99


## 3. Kiến thức nền

### 3.1 `list.pop(i)` và `list.append`

`append` thêm vào cuối `O(1)`. `pop(1)` xoá phần tử ở chỉ số 1 nhưng phải **dời** mọi phần tử phía sau sang trái ⇒ `O(n)`.

In [3]:
a = [2, 1, 3, 5]
loser = a.pop(1)       # xoá arr[1]
a.append(loser)        # đưa xuống cuối
print(a)

[2, 3, 5, 1]


### 3.2 `collections.deque` — hàng đợi hai đầu

`popleft()`, `appendleft()`, `append()`, `pop()` đều `O(1)`. Để mô phỏng, giữ nhà vô địch ở **biến riêng** và các người chờ trong `deque`: lấy người thách đấu bằng `popleft()`, người thua `append()` xuống cuối.

In [4]:
q = deque([1, 3, 5, 4])
champ = 2
challenger = q.popleft()
winner, loser = max(champ, challenger), min(champ, challenger)
q.append(loser)
print("vô địch:", winner, "| hàng chờ:", list(q))

vô địch: 2 | hàng chờ: [3, 5, 4, 1]


### 3.3 `max()` và chỉ số của phần tử lớn nhất

`max(arr)` trả giá trị lớn nhất `O(n)`; `arr.index(max(arr))` cho vị trí của nó.

In [5]:
arr = [2, 1, 3, 5, 4, 6, 7]
print(max(arr), arr.index(max(arr)))

7 6


### 3.4 Vòng `for` với `break` và mệnh đề `else`

Trong Python, `for ... else`: khối `else` chạy khi vòng lặp **kết thúc tự nhiên** (không gặp `break`). Hữu ích để phân biệt "tìm thấy người thắng giữa chừng" và "duyệt hết mảng".

In [6]:
for x in [1, 2, 3]:
    if x == 5:
        print("gặp 5, break")
        break
else:
    print("không gặp break -> khối else chạy")

không gặp break -> khối else chạy


### 3.5 `bisect` — tìm kiếm nhị phân trên mảng đã sắp xếp

`bisect_left(a, x)` trả về vị trí đầu tiên mà `a[i] ≥ x`. Dùng ở bài tập 5 (trả lời nhiều truy vấn `k`).

In [7]:
a = [1, 3, 3, 7, 10]
print(bisect.bisect_left(a, 3), bisect.bisect_left(a, 4), bisect.bisect_left(a, 11))

1 3 5


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1 nhưng chỉ nhìn **nhà vô địch** và **người thách đấu**:

- Vòng 1: vô địch 2, thách đấu 1 → 2 thắng (chuỗi 1).
- Vòng 2: vô địch 2, thách đấu **3** → 3 thắng, chuỗi **reset về 1**.
- Vòng 3: vô địch 3, thách đấu 5 → 5 thắng, chuỗi 1.
- Vòng 4: vô địch 5, thách đấu 4 → 5 thắng, chuỗi 2 = k ⇒ dừng, đáp án 5.

Quan sát: người thách đấu ở vòng `i` **luôn là `arr[i]`** (theo đúng thứ tự ban đầu) chừng nào ta chưa đi hết mảng gốc — vì những người thua bị xếp xuống **sau** tất cả các phần tử chưa thi đấu. Vậy trong `n − 1` vòng đầu, ta chỉ cần duyệt `arr[1], arr[2], …` theo thứ tự!

Và sau vòng `n − 1`, nhà vô địch là `max(arr)` — không ai thắng được nữa.

In [8]:
arr, k = [2, 1, 3, 5, 4, 6, 7], 2
champ, streak = arr[0], 0
for i in range(1, len(arr)):
    old = champ
    if arr[i] > champ:
        champ, streak = arr[i], 1
    else:
        streak += 1
    print(f"vòng {i}: vô địch cũ {old}, thách đấu arr[{i}]={arr[i]} -> thắng {champ}, chuỗi {streak}")
    if streak == k:
        print("=> đáp án", champ)
        break

vòng 1: vô địch cũ 2, thách đấu arr[1]=1 -> thắng 2, chuỗi 1
vòng 2: vô địch cũ 2, thách đấu arr[2]=3 -> thắng 3, chuỗi 1
vòng 3: vô địch cũ 3, thách đấu arr[3]=5 -> thắng 5, chuỗi 1
vòng 4: vô địch cũ 5, thách đấu arr[4]=4 -> thắng 5, chuỗi 2
=> đáp án 5


## 5. Cách 1 — Brute Force / baseline: mô phỏng đúng luật bằng `list`

Làm y hệt đề: so `arr[0]` và `arr[1]`, đổi chỗ nếu cần, `pop(1)` người thua rồi `append` xuống cuối, đếm chuỗi thắng. Để không chạy `10⁹` vòng, ta thêm một quan sát an toàn: nếu `k ≥ n − 1` thì người thắng chắc chắn là `max(arr)` (chứng minh ở mục 10) — nhưng ở baseline ta **giới hạn** số vòng bằng `max_rounds` để thấy mô phỏng thuần tuý chậm thế nào.

In [9]:
def simulate_list(arr, k, max_rounds=10**7):
    a = list(arr)
    streak, rounds = 0, 0
    while True:
        rounds += 1
        if rounds > max_rounds:
            return None                 # bỏ cuộc: quá nhiều vòng
        if a[0] < a[1]:
            a[0], a[1] = a[1], a[0]     # người thắng lên vị trí 0
            streak = 1
        else:
            streak += 1
        a.append(a.pop(1))              # người thua xuống cuối
        if streak == k:
            return a[0]

for arr, k, want in examples + more[:1]:
    print(arr, k, "->", simulate_list(arr, k), "mong đợi", want)

[2, 1, 3, 5, 4, 6, 7] 2 -> 5 mong đợi 5
[3, 2, 1] 10 -> 3 mong đợi 3
[1, 9, 8, 2, 3, 7, 6, 4, 5] 7 -> 9 mong đợi 9


Cẩn thận ở vòng 1: chuỗi của `arr[0]` ban đầu là 0; nếu `arr[0]` thắng thì thành 1 (dòng `streak += 1`), nếu thua thì người mới có chuỗi 1. Cả hai nhánh đều đúng.

**Độ phức tạp:** mỗi vòng `O(n)` (do `pop(1)`), số vòng có thể tới `k` ⇒ `O(k · n)`. Với `k = 10⁹` là không thể. Thử với ví dụ `k = 10⁹`:

In [10]:
arr, k, want = more[1]
t0 = time.perf_counter()
res = simulate_list(arr, k, max_rounds=200000)
print("Kết quả:", res, f"(bỏ cuộc sau 200000 vòng, {time.perf_counter() - t0:.2f}s) — muốn mô phỏng 10^9 vòng là bất khả")

Kết quả: None (bỏ cuộc sau 200000 vòng, 0.02s) — muốn mô phỏng 10^9 vòng là bất khả


**Nút thắt:** (1) `pop(1)` tốn `O(n)`; (2) mô phỏng tiếp cả khi nhà vô địch đã là `max(arr)` — lúc đó kết quả đã rõ.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — `max` là điểm dừng.** Khi nhà vô địch là `max(arr)`, nó thắng mọi vòng sau ⇒ đáp án là nó, bất kể `k`. Mà `max(arr)` lên ngôi sau tối đa `n − 1` vòng.

**Insight 2 — Người thách đấu đi theo thứ tự mảng gốc.** Trong `n − 1` vòng đầu, người thách đấu ở vòng `i` là `arr[i]`. Không cần hàng đợi, chỉ cần duyệt mảng.

**Quy tắc:**
- **Nếu** `arr[i] > champ` **thì** `champ = arr[i]`, `streak = 1`.
- **Ngược lại** `streak += 1`.
- **Nếu** `streak == k` **thì** trả `champ`.
- **Nếu** duyệt hết mảng mà chưa dừng **thì** `champ == max(arr)` ⇒ trả `champ`.

**Invariant:** sau khi xét `arr[i]`, `champ = max(arr[0..i])` và `streak` = số vòng liên tiếp `champ` đã thắng tính tới vòng `i`.

## 7. Các cách tiếp theo

### Cách 2 — Mô phỏng bằng `deque` + dừng sớm khi gặp `max`

Giữ `champ` ở biến riêng, hàng chờ là `deque`. Mỗi vòng `O(1)`. Dừng khi chuỗi đạt `k` **hoặc** `champ == max(arr)`.

In [11]:
def simulate_deque(arr, k):
    mx = max(arr)
    champ, q = arr[0], deque(arr[1:])
    streak = 0
    while streak < k and champ != mx:
        challenger = q.popleft()
        if challenger > champ:
            q.append(champ)
            champ, streak = challenger, 1
        else:
            q.append(challenger)
            streak += 1
    return champ

for arr, k, want in examples + more:
    print(arr, k, "->", simulate_deque(arr, k), "mong đợi", want)

[2, 1, 3, 5, 4, 6, 7] 2 -> 5 mong đợi 5
[3, 2, 1] 10 -> 3 mong đợi 3
[1, 9, 8, 2, 3, 7, 6, 4, 5] 7 -> 9 mong đợi 9
[1, 11, 22, 33, 44, 55, 66, 77, 88, 99] 1000000000 -> 99 mong đợi 99


Chú ý điều kiện dừng `champ != mx`: nếu `champ` đã là `max` thì dù chuỗi chưa đủ `k`, kết quả vẫn là `champ`.

Code nộp LeetCode cho cách 2:

In [12]:
class SolutionDeque:
    def getWinner(self, arr: List[int], k: int) -> int:
        mx = max(arr)
        champ, q, streak = arr[0], deque(arr[1:]), 0
        while streak < k and champ != mx:
            c = q.popleft()
            if c > champ:
                q.append(champ)
                champ, streak = c, 1
            else:
                q.append(c)
                streak += 1
        return champ

print([SolutionDeque().getWinner(a, k) for a, k, _ in examples + more])

[5, 3, 9, 99]


**Độ phức tạp:** trước khi `max` lên ngôi có tối đa `n − 1` vòng; sau đó dừng ngay. Vậy Time `O(n)`, Space `O(n)` cho `deque`.

### Cách 3 — Một lượt duyệt, `O(1)` bộ nhớ

Từ insight 2, bỏ luôn `deque`. Giải thích từng dòng:

- `champ = arr[0]`, `streak = 0`: chưa thi đấu vòng nào.
- Duyệt `x` trong `arr[1:]`: `x` là người thách đấu.
- Thắng mới ⇒ reset chuỗi **về 1** (vừa thắng một vòng).
- Đạt `k` ⇒ `break`.
- Hết vòng lặp mà chưa đạt ⇒ `champ` là `max` ⇒ vẫn trả `champ`.

In [13]:
def one_pass(arr, k):
    champ, streak = arr[0], 0
    for x in arr[1:]:
        if x > champ:
            champ, streak = x, 1
        else:
            streak += 1
        if streak == k:
            break
    return champ

for arr, k, want in examples + more:
    print(arr, k, "->", one_pass(arr, k), "mong đợi", want)

[2, 1, 3, 5, 4, 6, 7] 2 -> 5 mong đợi 5
[3, 2, 1] 10 -> 3 mong đợi 3
[1, 9, 8, 2, 3, 7, 6, 4, 5] 7 -> 9 mong đợi 9
[1, 11, 22, 33, 44, 55, 66, 77, 88, 99] 1000000000 -> 99 mong đợi 99


In [14]:
class SolutionOnePass:
    def getWinner(self, arr: List[int], k: int) -> int:
        champ, streak = arr[0], 0
        for i in range(1, len(arr)):
            if arr[i] > champ:
                champ, streak = arr[i], 1
            else:
                streak += 1
            if streak == k:
                return champ
        return champ            # đã là max(arr)

print([SolutionOnePass().getWinner(a, k) for a, k, _ in examples + more])

[5, 3, 9, 99]


**Độ phức tạp:** Time `O(n)`, Space `O(1)` (dùng chỉ số thay cho `arr[1:]` để khỏi tạo bản sao).

### Cách 4 — Đi theo "kỷ lục" (next greater element)

Nhà vô địch chỉ đổi khi gặp một **kỷ lục mới** (phần tử lớn hơn mọi phần tử trước nó). Kỷ lục ở chỉ số `i` sẽ giữ ngôi tới kỷ lục kế tiếp ở chỉ số `j`, nên chuỗi thắng của nó là `(j − i − 1) + (1 nếu i > 0)` (thắng các phần tử ở giữa, cộng trận thắng lúc lên ngôi; `arr[0]` không có trận "lên ngôi"). Kỷ lục cuối cùng là `max`, chuỗi vô hạn. Đáp án = kỷ lục **đầu tiên** có chuỗi `≥ k`. Cách nhìn này giúp trả lời **nhiều truy vấn `k`** (bài tập 5).

In [15]:
def records(arr):
    """Danh sách (giá trị kỷ lục, chuỗi thắng tối đa); kỷ lục cuối có chuỗi vô hạn."""
    idx = [0]
    for i in range(1, len(arr)):
        if arr[i] > arr[idx[-1]]:
            idx.append(i)
    out = []
    for t, i in enumerate(idx):
        if t + 1 < len(idx):
            j = idx[t + 1]
            out.append((arr[i], (j - i - 1) + (1 if i > 0 else 0)))
        else:
            out.append((arr[i], float("inf")))
    return out

def by_records(arr, k):
    for val, streak in records(arr):
        if streak >= k:
            return val

print(records([2, 1, 3, 5, 4, 6, 7]))
for arr, k, want in examples + more:
    print(arr, k, "->", by_records(arr, k), "mong đợi", want)

[(2, 1), (3, 1), (5, 2), (6, 1), (7, inf)]
[2, 1, 3, 5, 4, 6, 7] 2 -> 5 mong đợi 5
[3, 2, 1] 10 -> 3 mong đợi 3
[1, 9, 8, 2, 3, 7, 6, 4, 5] 7 -> 9 mong đợi 9
[1, 11, 22, 33, 44, 55, 66, 77, 88, 99] 1000000000 -> 99 mong đợi 99


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (lưu danh sách kỷ lục).

## 8. TRACE từng bước

Bảng trạng thái cách 3 với `arr = [1,9,8,2,3,7,6,4,5]`, `k = 7`:

| i | arr[i] | champ trước | thắng | champ sau | streak |
|---|--------|-------------|-------|-----------|--------|
| 1 | 9 | 1 | 9 (mới) | 9 | 1 |
| 2 | 8 | 9 | 9 | 9 | 2 |
| 3 | 2 | 9 | 9 | 9 | 3 |
| 4 | 3 | 9 | 9 | 9 | 4 |
| 5 | 7 | 9 | 9 | 9 | 5 |
| 6 | 6 | 9 | 9 | 9 | 6 |
| 7 | 4 | 9 | 9 | 9 | **7 = k** → dừng |

Hàm `trace_one_pass` in lại bảng; hàm `trace_deque` in trạng thái hàng chờ thật để đối chiếu.

In [16]:
def trace_one_pass(arr, k):
    champ, streak = arr[0], 0
    print(f"{'i':>2} | arr[i] | champ trước | champ sau | streak")
    for i in range(1, len(arr)):
        before = champ
        if arr[i] > champ:
            champ, streak = arr[i], 1
        else:
            streak += 1
        print(f"{i:>2} | {arr[i]:>6} | {before:>11} | {champ:>9} | {streak:>6}" + ("  <- đạt k" if streak == k else ""))
        if streak == k:
            return champ
    print("duyệt hết mảng: champ là max")
    return champ

print("Kết quả:", trace_one_pass([1, 9, 8, 2, 3, 7, 6, 4, 5], 7))
print()
print("Kết quả:", trace_one_pass([3, 2, 1], 10))

 i | arr[i] | champ trước | champ sau | streak
 1 |      9 |           1 |         9 |      1
 2 |      8 |           9 |         9 |      2
 3 |      2 |           9 |         9 |      3
 4 |      3 |           9 |         9 |      4
 5 |      7 |           9 |         9 |      5
 6 |      6 |           9 |         9 |      6
 7 |      4 |           9 |         9 |      7  <- đạt k
Kết quả: 9

 i | arr[i] | champ trước | champ sau | streak
 1 |      2 |           3 |         3 |      1
 2 |      1 |           3 |         3 |      2
duyệt hết mảng: champ là max
Kết quả: 3


In [17]:
def trace_deque(arr, k):
    mx = max(arr)
    champ, q, streak, rnd = arr[0], deque(arr[1:]), 0, 0
    while streak < k and champ != mx:
        rnd += 1
        c = q.popleft()
        if c > champ:
            q.append(champ)
            champ, streak = c, 1
        else:
            q.append(c)
            streak += 1
        print(f"vòng {rnd}: mảng = {[champ] + list(q)} | thắng {champ} | chuỗi {streak}")
    print("dừng vì", "đạt k" if streak >= k else "champ là max")
    return champ

print("Kết quả:", trace_deque([2, 1, 3, 5, 4, 6, 7], 2))

vòng 1: mảng = [2, 3, 5, 4, 6, 7, 1] | thắng 2 | chuỗi 1
vòng 2: mảng = [3, 5, 4, 6, 7, 1, 2] | thắng 3 | chuỗi 1
vòng 3: mảng = [5, 4, 6, 7, 1, 2, 3] | thắng 5 | chuỗi 1
vòng 4: mảng = [5, 6, 7, 1, 2, 3, 4] | thắng 5 | chuỗi 2
dừng vì đạt k
Kết quả: 5


## 9. Code nộp LeetCode

Bản nộp chính — cách 3, đúng chữ ký `getWinner(self, arr: List[int], k: int) -> int`:

In [18]:
class Solution:
    def getWinner(self, arr: List[int], k: int) -> int:
        champ = arr[0]
        streak = 0
        for i in range(1, len(arr)):
            if arr[i] > champ:          # người thách đấu thắng: lên ngôi, chuỗi = 1
                champ = arr[i]
                streak = 1
            else:                       # nhà vô địch thắng thêm một vòng
                streak += 1
            if streak == k:
                return champ
        return champ                    # duyệt hết: champ = max(arr), thắng mãi

for arr, k, want in examples + more:
    got = Solution().getWinner(arr, k)
    print(arr, k, "->", got, "OK" if got == want else "SAI")

[2, 1, 3, 5, 4, 6, 7] 2 -> 5 OK
[3, 2, 1] 10 -> 3 OK
[1, 9, 8, 2, 3, 7, 6, 4, 5] 7 -> 9 OK
[1, 11, 22, 33, 44, 55, 66, 77, 88, 99] 1000000000 -> 99 OK


## 10. Vì sao thuật toán đúng

**Bổ đề (hình dạng mảng).** Sau vòng `i` (`1 ≤ i ≤ n − 1`), mảng có dạng
`[M_i, arr[i+1], arr[i+2], …, arr[n−1], (i phần tử đã thua theo thứ tự thua)]` với `M_i = max(arr[0..i])`.

*Chứng minh quy nạp:* vòng 1 so `arr[0]` với `arr[1]`, người lớn hơn là `M_1` đứng đầu, người thua xuống cuối — đúng dạng. Giả sử đúng sau vòng `i`; vòng `i+1` so `M_i` với `arr[i+1]` (phần tử thứ hai), người thắng là `max(M_i, arr[i+1]) = M_{i+1}`, người thua bị đưa ra **sau** phần tử thua cuối cùng — đúng dạng. ∎

**Hệ quả 1:** người thách đấu ở vòng `i` là `arr[i]` ⇒ cách 3 mô phỏng **chính xác** `n − 1` vòng đầu (cùng nhà vô địch, cùng chuỗi thắng).

**Hệ quả 2:** sau vòng `n − 1`, `M_{n−1} = max(arr)` đứng đầu; vì các số phân biệt, nó thắng mọi vòng sau ⇒ nếu trò chơi chưa kết thúc trong `n − 1` vòng đầu thì người thắng là `max(arr)`, đúng giá trị `champ` khi vòng lặp kết thúc.

**Chuỗi thắng:** người thách đấu thắng thì vừa thắng đúng 1 vòng (`streak = 1`); nhà vô địch thắng thì chuỗi tăng 1 — khớp định nghĩa "`k` vòng liên tiếp".

## 11. Phân tích Time / Space

Gọi `n = len(arr)`.

| Cách | Time | Space | Ghi chú |
|------|------|-------|---------|
| 1. Mô phỏng `list` | `O(min(k, ∞) · n)` — vô hạn thực tế khi `k` lớn | `O(n)` | `pop(1)` tốn `O(n)` |
| 2. `deque` + dừng ở max | `O(n)` | `O(n)` | ≤ `n − 1` vòng trước khi gặp max |
| 3. Một lượt duyệt | `O(n)` | `O(1)` | bản nộp chính |
| 4. Theo kỷ lục | `O(n)` | `O(n)` | hợp cho nhiều truy vấn `k` |

Best case cách 3: `O(min(n, k))` — dừng ngay khi đạt `k` (ví dụ `arr[0]` lớn và `k = 1`). Worst case: duyệt hết `n − 1` phần tử.

## 12. Bộ test

Ví dụ đề + ví dụ thường gặp, edge case (`n = 2`, `k = 1`, `k = 10⁹`, mảng tăng, mảng giảm, max ở đầu/cuối), random seed cố định so với mô phỏng `list` (với `k` nhỏ để mô phỏng được), và test lớn `n = 10⁵`.

In [19]:
METHODS = {
    "deque": simulate_deque,
    "SolutionDeque": lambda a, k: SolutionDeque().getWinner(a, k),
    "one_pass": one_pass,
    "SolutionOnePass": lambda a, k: SolutionOnePass().getWinner(a, k),
    "records": by_records,
    "Solution": lambda a, k: Solution().getWinner(a, k),
}
edge = [([1, 2], 1, 2), ([2, 1], 1, 2), ([1, 2], 10**9, 2), ([5, 4, 3, 2, 1], 1, 5),
        ([1, 2, 3, 4, 5], 1, 2), ([1, 2, 3, 4, 5], 2, 5), ([1, 3, 2, 5, 4], 2, 3),
        ([3, 1, 2], 2, 3), ([1, 3, 2], 1, 3), ([10, 1, 2, 3], 3, 10)]
for arr, k, want in examples + more + edge:
    if k <= 10**4:
        assert simulate_list(arr, k) == want, (arr, k)
    for name, f in METHODS.items():
        got = f(arr, k)
        assert got == want, (name, arr, k, got, want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [20]:
rng = random.Random(1535)
for _ in range(500):
    n = rng.randint(2, 40)
    arr = rng.sample(range(1, 10**6 + 1), n)
    k = rng.choice([rng.randint(1, n + 3), rng.randint(1, 3 * n), 10**9])
    ref = simulate_list(arr, k) if k <= 3 * n + 5 else max(arr)
    for name, f in METHODS.items():
        assert f(arr, k) == ref, (name, arr, k)
print("500 mảng ngẫu nhiên (seed 1535): mọi cách khớp mô phỏng")

500 mảng ngẫu nhiên (seed 1535): mọi cách khớp mô phỏng


In [21]:
rng2 = random.Random(9)
big = rng2.sample(range(1, 10**6 + 1), 10**5)
inc = list(range(1, 10**5 + 1))
for name in ["deque", "one_pass", "records", "Solution"]:
    t0 = time.perf_counter()
    res = [METHODS[name](big, 10**9), METHODS[name](big, 50), METHODS[name](inc, 10**5)]
    print(f"{name:9s}: {res} ({time.perf_counter() - t0:.3f}s)")
    assert res[0] == max(big) and res[2] == 10**5

deque    : [999979, 997481, 100000] (0.025s)


one_pass : [999979, 997481, 100000] (0.014s)


records  : [999979, 997481, 100000] (0.054s)


Solution : [999979, 997481, 100000] (0.015s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Reset chuỗi về 0 khi có nhà vô địch mới

Người thách đấu vừa **thắng một vòng** nên chuỗi phải là 1. Đặt `streak = 0` làm ta cần thêm một vòng ⇒ chọn sai người.

In [22]:
def wrong_reset_zero(arr, k):
    champ, streak = arr[0], 0
    for x in arr[1:]:
        if x > champ:
            champ, streak = x, 0       # SAI: phải là 1
        else:
            streak += 1
        if streak == k:
            return champ
    return champ

for arr, k, want in examples + [([1, 3, 2, 5, 4], 2, 3), ([1, 11, 22, 33], 1, 11)]:
    print(arr, k, "| sai:", wrong_reset_zero(arr, k), "| đúng:", want)

[2, 1, 3, 5, 4, 6, 7] 2 | sai: 7 | đúng: 5
[3, 2, 1] 10 | sai: 3 | đúng: 3
[1, 3, 2, 5, 4] 2 | sai: 5 | đúng: 3
[1, 11, 22, 33] 1 | sai: 33 | đúng: 11


Sửa: `champ, streak = x, 1`.

### Lỗi 2 — Luôn trả về `max(arr)`

Đúng khi `k` lớn, sai khi `k` nhỏ: trò chơi có thể kết thúc trước khi `max` lên ngôi.

In [23]:
def wrong_always_max(arr, k):
    return max(arr)                    # SAI khi k nhỏ

for arr, k, want in examples + more:
    print(arr, k, "| sai:", wrong_always_max(arr, k), "| đúng:", want)

[2, 1, 3, 5, 4, 6, 7] 2 | sai: 7 | đúng: 5
[3, 2, 1] 10 | sai: 3 | đúng: 3
[1, 9, 8, 2, 3, 7, 6, 4, 5] 7 | sai: 9 | đúng: 9
[1, 11, 22, 33, 44, 55, 66, 77, 88, 99] 1000000000 | sai: 99 | đúng: 99


Sửa: chỉ trả `max(arr)` khi duyệt hết mảng mà chưa có ai đạt `k` (hoặc khi `k ≥ n − 1`).

### Lỗi 3 — Duyệt hết mảng thì trả `-1` / phần tử cuối

Quên rằng "hết mảng" nghĩa là `champ` đã là `max` và sẽ thắng mãi.

In [24]:
def wrong_end(arr, k):
    champ, streak = arr[0], 0
    for x in arr[1:]:
        if x > champ:
            champ, streak = x, 1
        else:
            streak += 1
        if streak == k:
            return champ
    return arr[-1]                     # SAI: phải trả champ

for arr, k, want in examples + more:
    print(arr, k, "| sai:", wrong_end(arr, k), "| đúng:", want)

[2, 1, 3, 5, 4, 6, 7] 2 | sai: 5 | đúng: 5
[3, 2, 1] 10 | sai: 1 | đúng: 3
[1, 9, 8, 2, 3, 7, 6, 4, 5] 7 | sai: 9 | đúng: 9
[1, 11, 22, 33, 44, 55, 66, 77, 88, 99] 1000000000 | sai: 99 | đúng: 99


Sửa: `return champ` sau vòng lặp.

### Lỗi 4 — Mô phỏng nguyên `k` vòng với `k = 10⁹`

Kết quả đúng về lý thuyết nhưng chạy mãi không xong (TLE). Ta đếm số vòng mà cách mô phỏng "ngây thơ" sẽ phải chạy, thay vì chạy thật:

In [25]:
def naive_rounds_needed(arr, k):
    """Số vòng trò chơi thật sự kéo dài (tính bằng công thức, không mô phỏng)."""
    champ, streak = arr[0], 0
    for i in range(1, len(arr)):
        if arr[i] > champ:
            champ, streak = arr[i], 1
        else:
            streak += 1
        if streak == k:
            return i
    return (len(arr) - 1) + (k - streak)

arr, k, _ = more[1]
print("Số vòng mô phỏng ngây thơ cần chạy:", naive_rounds_needed(arr, k), "-> không thể; cách 3 chỉ duyệt", len(arr) - 1, "phần tử")

Số vòng mô phỏng ngây thơ cần chạy: 1000000008 -> không thể; cách 3 chỉ duyệt 9 phần tử


Sửa: dừng ngay khi nhà vô địch là `max(arr)` (cách 2) hoặc dùng cách 3.

### Lỗi 5 — Đếm chuỗi bắt đầu từ 1 cho `arr[0]` trước khi thi đấu

Khởi tạo `streak = 1` cho `arr[0]` như thể nó đã thắng một vòng ⇒ dừng sớm một vòng.

In [26]:
def wrong_init_one(arr, k):
    champ, streak = arr[0], 1          # SAI: arr[0] chưa thắng vòng nào
    if streak == k:
        return champ
    for x in arr[1:]:
        if x > champ:
            champ, streak = x, 1
        else:
            streak += 1
        if streak == k:
            return champ
    return champ

for arr, k, want in examples + [([1, 2, 3], 1, 2), ([2, 1, 3], 2, 3)]:
    print(arr, k, "| sai:", wrong_init_one(arr, k), "| đúng:", want)

[2, 1, 3, 5, 4, 6, 7] 2 | sai: 2 | đúng: 5
[3, 2, 1] 10 | sai: 3 | đúng: 3
[1, 2, 3] 1 | sai: 1 | đúng: 2
[2, 1, 3] 2 | sai: 2 | đúng: 3


Sửa: `streak = 0` ban đầu.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Mô phỏng cho tới khi …" với tham số khổng lồ (`k ≤ 10⁹`)** ⇒ tìm trạng thái **ổn định** (ở đây: `max` đứng đầu) để dừng sớm.
- Hàng đợi mà phần tử thua bị đẩy xuống cuối ⇒ trong một "vòng" đầu, các phần tử được gặp **theo thứ tự ban đầu** ⇒ thay mô phỏng bằng duyệt mảng.
- Theo dõi **"nhà vô địch hiện tại" + "chuỗi"** là pattern quen thuộc (giống đếm chuỗi dài nhất, Boyer–Moore voting).
- Gợi ý của đề "If k ≥ arr.length return the max element" chính là insight 1.
- Bài cùng họ: *Dota2 Senate* (649), *Time Needed to Buy Tickets* (2073), *Find the Winner of the Circular Game* (1823), *Next Greater Element* (496).

## 15. Biến thể của bài

1. **Số nhỏ hơn thắng:** đổi dấu so sánh; điểm dừng là `min(arr)`.
2. **Có phần tử trùng nhau** (hoà thì người đang giữ ngôi thắng): khi gặp hai `max` bằng nhau, `max` đầu tiên thắng mãi — thuật toán vẫn đúng nếu dùng `>` (không phải `≥`).
3. **Trả về số vòng đã chơi** (bài tập 4).
4. **Nhiều truy vấn `k`** trên cùng mảng: tiền xử lý danh sách kỷ lục + chuỗi thắng, trả lời mỗi truy vấn bằng tìm kiếm nhị phân (bài tập 5).
5. **Người thua về đầu hàng chờ thay vì cuối** hoặc **người thắng phải xuống cuối**: phá vỡ bổ đề hình dạng mảng — phải mô phỏng bằng `deque` và tìm chu kỳ.

In [27]:
def variant_min_wins(arr, k):
    champ, streak = arr[0], 0
    for x in arr[1:]:
        if x < champ:
            champ, streak = x, 1
        else:
            streak += 1
        if streak == k:
            return champ
    return champ

def simulate_min(arr, k, cap=10**5):
    a, streak = list(arr), 0
    for _ in range(cap):
        if a[0] > a[1]:
            a[0], a[1] = a[1], a[0]; streak = 1
        else:
            streak += 1
        a.append(a.pop(1))
        if streak == k:
            return a[0]
    return a[0]

rngv = random.Random(15)
for _ in range(200):
    arr = rngv.sample(range(1, 100), rngv.randint(2, 12))
    k = rngv.randint(1, 15)
    assert variant_min_wins(arr, k) == simulate_min(arr, k)
print("Biến thể 'số nhỏ thắng': khớp mô phỏng; ví dụ", variant_min_wins([2, 1, 3, 5, 4, 6, 7], 2))

Biến thể 'số nhỏ thắng': khớp mô phỏng; ví dụ 1


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Mô phỏng `list` | làm đúng luật | `O(k·n)` | `O(n)` | kiểm chứng với `k` nhỏ |
| 2. `deque` + dừng ở max | mô phỏng `O(1)`/vòng, dừng sớm | `O(n)` | `O(n)` | khi luật phức tạp hơn, vẫn muốn mô phỏng |
| 3. Một lượt duyệt | người thách đấu = `arr[i]` | `O(n)` | `O(1)` | **bản nộp chính** |
| 4. Kỷ lục | chuỗi thắng của mỗi kỷ lục | `O(n)` | `O(n)` | nhiều truy vấn `k` |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `getWinner([1,25,35,42,68,70], 1)` trả về gì? Còn với `k = 2`?

**Bài 2 (tính tay):** Với `arr = [4, 2, 6, 1, 3, 5, 9, 7]`, liệt kê danh sách kỷ lục kèm chuỗi thắng (theo công thức cách 4) và đáp án cho `k = 1, 2, 3, 4`.

**Bài 3 (điền TODO):** Hoàn thiện `ex3_winner` (một lượt duyệt).

**Bài 4 (tự viết lại):** Viết `ex4_rounds(arr, k)` trả về **số vòng** đã chơi khi trò chơi kết thúc (không mô phỏng `k` vòng).

**Bài 5 (mini challenge):** Viết `ex5_multi(arr, ks)` trả về danh sách người thắng cho từng `k` trong `ks` (có thể tới `10⁵` truy vấn) trong `O((n + q) log n)`.

In [28]:
# Bài 3 — điền TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_winner(arr: List[int], k: int) -> int:
    champ, streak = arr[0], 0
    for i in range(1, len(arr)):
        # TODO: cập nhật champ/streak; nếu streak == k thì trả champ
        pass
    return champ

In [29]:
# Bài 4 & 5 — khung tự viết (chỉ định nghĩa, KHÔNG gọi)
def ex4_rounds(arr: List[int], k: int) -> int:
    # TODO
    raise NotImplementedError

def ex5_multi(arr: List[int], ks: List[int]) -> List[int]:
    # TODO
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Vòng 1: 1 vs 25. (2) 25 thắng, chuỗi 1. (3) `k = 1` → 25; `k = 2`: 25 thua 35 ở vòng 2, … chuỗi luôn reset → hết mảng → 70.

**Bài 2.** (1) Kỷ lục: 4 (i=0), 6 (i=2), 9 (i=6). (2) Chuỗi: 4 → `2−0−1+0 = 1`; 6 → `6−2−1+1 = 4`; 9 → ∞. (3) `k=1` → 4; `k=2,3,4` → 6.

**Bài 3.** (1) `if arr[i] > champ`. (2) Hai nhánh: `champ, streak = arr[i], 1` / `streak += 1`. (3) Kiểm tra `streak == k` sau mỗi vòng.

**Bài 4.** (1) Nếu dừng ở chỉ số `i` thì đã chơi `i` vòng. (2) Nếu duyệt hết: đã chơi `n − 1` vòng, `max` cần thêm `k − streak` vòng. (3) Đối chiếu với mô phỏng `list` đếm vòng cho `k` nhỏ.

**Bài 5.** (1) Tính danh sách kỷ lục `(val, streak)`. (2) Lập `best[t] = max(streak[0..t])` — mảng không giảm. (3) Với mỗi `k`: `t = bisect_left(best, k)`, đáp án là `val[t]`.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1
a1 = (Solution().getWinner([1, 25, 35, 42, 68, 70], 1), Solution().getWinner([1, 25, 35, 42, 68, 70], 2))
print("Bài 1:", a1)
assert a1 == (25, 70)

# Bài 2
arr2 = [4, 2, 6, 1, 3, 5, 9, 7]
rec = records(arr2)
ans2 = [Solution().getWinner(arr2, k) for k in (1, 2, 3, 4)]
print("Bài 2: kỷ lục", rec, "| đáp án k=1..4:", ans2)
assert rec == [(4, 1), (6, 4), (9, float("inf"))] and ans2 == [4, 6, 6, 6]

Bài 1: (25, 70)
Bài 2: kỷ lục [(4, 1), (6, 4), (9, inf)] | đáp án k=1..4: [4, 6, 6, 6]


In [31]:
# Bài 3
def ex3_winner_sol(arr, k):
    champ, streak = arr[0], 0
    for i in range(1, len(arr)):
        if arr[i] > champ:
            champ, streak = arr[i], 1
        else:
            streak += 1
        if streak == k:
            return champ
    return champ

rng3 = random.Random(3)
for _ in range(300):
    arr = rng3.sample(range(1, 1000), rng3.randint(2, 30))
    k = rng3.randint(1, 40)
    assert ex3_winner_sol(arr, k) == simulate_list(arr, k)
print("Bài 3: OK")

Bài 3: OK


In [32]:
# Bài 4
def ex4_rounds_sol(arr, k):
    champ, streak = arr[0], 0
    for i in range(1, len(arr)):
        if arr[i] > champ:
            champ, streak = arr[i], 1
        else:
            streak += 1
        if streak == k:
            return i
    return (len(arr) - 1) + (k - streak)

def rounds_by_sim(arr, k):
    a, streak, rounds = list(arr), 0, 0
    while True:
        rounds += 1
        if a[0] < a[1]:
            a[0], a[1] = a[1], a[0]; streak = 1
        else:
            streak += 1
        a.append(a.pop(1))
        if streak == k:
            return rounds

rng4 = random.Random(4)
for _ in range(300):
    arr = rng4.sample(range(1, 1000), rng4.randint(2, 20))
    k = rng4.randint(1, 30)
    assert ex4_rounds_sol(arr, k) == rounds_by_sim(arr, k)
print("Bài 4: OK, ví dụ 1 chơi", ex4_rounds_sol([2, 1, 3, 5, 4, 6, 7], 2), "vòng")

Bài 4: OK, ví dụ 1 chơi 4 vòng


In [33]:
# Bài 5
def ex5_multi_sol(arr, ks):
    rec = records(arr)
    vals = [v for v, _ in rec]
    best, cur = [], 0
    for _, s in rec:
        cur = max(cur, s)
        best.append(cur)
    return [vals[bisect.bisect_left(best, k)] for k in ks]

rng5 = random.Random(5)
for _ in range(100):
    arr = rng5.sample(range(1, 10**6), rng5.randint(2, 200))
    ks = [rng5.randint(1, 250) for _ in range(30)] + [10**9]
    assert ex5_multi_sol(arr, ks) == [Solution().getWinner(arr, k) for k in ks]
print("Bài 5: OK, ví dụ", ex5_multi_sol([2, 1, 3, 5, 4, 6, 7], [1, 2, 3, 10**9]))

Bài 5: OK, ví dụ [2, 5, 7, 7]


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def getWinner(self, arr, k):
        champ, streak = arr[0], 0
        for x in arr[1:]:
            if x > champ: champ, streak = x, 1   # lên ngôi = đã thắng 1 vòng
            else:         streak += 1
            if streak == k: return champ
        return champ                              # hết mảng -> champ = max, thắng mãi
```

**Template "mô phỏng có trạng thái ổn định":** tìm thời điểm hệ "đóng băng" (phần tử mạnh nhất lên đầu) → mô phỏng tới đó rồi trả lời ngay; nếu thứ tự gặp là thứ tự ban đầu thì thay hàng đợi bằng duyệt mảng.

**Checklist trước khi submit:**
- [ ] `streak` khởi tạo `0`; nhà vô địch mới → `streak = 1`.
- [ ] Kiểm tra `streak == k` sau **mỗi** vòng (kể cả vòng vừa đổi ngôi, khi `k = 1`).
- [ ] Sau vòng lặp trả `champ` (chính là `max`), không trả `-1`.
- [ ] Không mô phỏng `k` vòng khi `k` tới `10⁹`.
- [ ] Thử nhanh: `[2,1,3,5,4,6,7], 2 → 5`; `[3,2,1], 10 → 3`; `[1,2], 1 → 2`.